# RFROM v2.3 → virtual Icechunk: smoke test

GitHub issue #17. Run this **before** building the real store, and again after
any change to `build_icechunk.py`. It builds a small store into a **local
temporary repository** — nothing is written to the NODD bucket — and then reads
it back the way a consumer will and compares the values against the source
netCDFs.

**What a virtual store is.** It holds Zarr metadata plus byte-range references.
The science data stays in the netCDFs in `gs://noaa-oar-rfrom/netcdf/v2.3/` and
is never copied. The whole 526 GB product becomes a store of a few MB. The price
is that nothing can be rechunked or recompressed: the store inherits the chunk
grid of the files exactly.

**The one thing that makes or breaks this build.** Every file feeding one Zarr
array must share one chunk grid, because Zarr has no variable-length chunks.
That single rule is why the netCDF layout had to change (issue #17):

1. each variable must be **one continuous series** of files — temperature could
   not stay split into a stable record plus a realtime one, because the merged
   time axis would need a 70-long chunk in the *middle*;
2. the **final short block must keep the full 100-step time chunk**, written with
   an unlimited time dimension so HDF5 pads the edge chunk.

The cells below demonstrate both, on the real published files.

In [ ]:
# %pip install -qU icechunk virtualizarr xarray zarr obstore gcsfs h5py numpy pandas

In [ ]:
import sys, os, time, shutil, copy
sys.path.insert(0, os.path.abspath(".."))     # build_icechunk.py lives at the repo root

import numpy as np, pandas as pd, xarray as xr, zarr, icechunk as ic, virtualizarr
import build_icechunk as B

for m in (ic, virtualizarr, xr, zarr):
    print(f"{m.__name__:<12} {m.__version__}")
print(f"{'python':<12} {sys.version.split()[0]}")

zarr.config.set({"async.concurrency": B.ZARR_CONCURRENCY})   # the default of 10 is the
                                                             # usual reason a store feels slow

## Configuration

`SCRATCH` only holds the throwaway repository. `LOCAL_BLOCK_DIR` is the escape
hatch for rehearsing a block that has been **built but not yet uploaded** — point
it at `nodd.py`'s output directory to check a rebuilt tail before it goes to the
bucket, or leave it `None` to use only what is published.

In [ ]:
STORE = "rfrom_v23"
STREAM = "temp_error"                 # one stream is enough to exercise every code path
N_FULL_BLOCKS = 2                     # published 100-step blocks to include (keep small)
SCRATCH = os.environ.get("NODD_SCRATCH_DIR", "/tmp/icechunk-smoke")
LOCAL_BLOCK_DIR = None                # e.g. f"{SCRATCH}/nodd"  -- a locally built tail block

cfg = copy.deepcopy(B.STORES[STORE])
cfg["variables"] = {STREAM: cfg["variables"][STREAM]}
VAR = cfg["variables"][STREAM]
print(f"{STORE}: {VAR} from gs://{cfg['bucket']}/{cfg['netcdf_prefix']}/{STREAM}/")

## 1. Discover the source files

Source and destination are two independent configurations. This cell touches only
the source, and reads it **anonymously** — the way a consumer will, not with the
credentials that write the store.

In [ ]:
urls = B.list_stream_files(cfg, STREAM)
print(f"{len(urls)} published files")
for u in urls[:2] + ["   ..."] + urls[-2:]:
    print("  ", u.split("/")[-1] if u.startswith("gs") else u)

## 2. Read the chunk grid of each file

This is the step that decides whether a store is possible at all. Only headers
are read — no science data moves. Watch the **time chunk** column: every file
must show the same number.

In [ ]:
registry = B.source_registry(cfg["bucket"])

selected = urls[-(N_FULL_BLOCKS + 1):]        # the last few, including the tail
if LOCAL_BLOCK_DIR:
    local = sorted(f for f in os.listdir(LOCAL_BLOCK_DIR) if f.endswith(".nc"))
    selected = urls[-(N_FULL_BLOCKS + 1):-1] + [f"file://{os.path.join(LOCAL_BLOCK_DIR, local[-1])}"]
    registry = B.source_registry(cfg["bucket"])
    from obstore.store import LocalStore
    registry.register("file://", LocalStore())

t0 = time.time()
vds = B.open_virtual_files(selected, registry, max_workers=4)
print(f"read {len(vds)} headers in {time.time() - t0:.0f}s\n")

print(f"{'file':<48}{'steps':>7}{'time chunk':>12}")
for u, v in zip(selected, vds):
    a = v[VAR].data
    print(f"{u.split('/')[-1]:<48}{a.shape[0]:>7}{a.chunks[0]:>12}")

## 3. The constraint, demonstrated

If the time chunks above are not all equal, the store cannot be built — and this
is exactly what the **published** tail block looks like, because it was written
before issue #17 with its time chunk shrunk to fit (19 instead of 100).

`build_icechunk.concat_virtual` refuses it by name and says how to fix it. If you
pointed `LOCAL_BLOCK_DIR` at a rebuilt tail, this cell succeeds instead.

In [ ]:
try:
    arr, times, attrs = B.concat_virtual(vds, VAR)
    print("concatenated:", arr.shape, "chunks", arr.chunks)
    print(f"time {times[0].date()} -> {times[-1].date()}")
    n_chunks = -(-arr.shape[0] // arr.chunks[0])
    print(f"{arr.shape[0]} steps over {n_chunks} chunks of {arr.chunks[0]} "
          f"(last one partial: {arr.shape[0] - (n_chunks - 1) * arr.chunks[0]} steps)")
except ValueError as exc:
    print("REJECTED:\n"); print(exc)
    print("\n-> rebuild that block with:  python nodd.py --stream", STREAM,
          "--blocks", len(urls) - 1)
    print("   then set LOCAL_BLOCK_DIR above and re-run from cell 2.")
    raise

Why this needs `concat_virtual` rather than `open_virtual_mfdataset`: VirtualiZarr
requires **every** input to be an exact multiple of the chunk length on the concat
axis, including the last one. RFROM is 1719 steps with a 100-step chunk, and no
publishing schedule will ever make that divide. A trailing partial chunk is
perfectly legal Zarr — edge chunks are stored full size and cropped on read — so
`concat_virtual` joins the chunk manifests directly, after checking the two
conditions Zarr does impose (one chunk shape everywhere; only the last file short).

## 4. Assemble the dataset

One group, one time axis, the science variables virtual and the coordinates
materialized. `data_mode` is added here — an int8 CF flag marking which weeks are
provisional realtime — rather than being carried in the netCDFs, so that promoting
realtime weeks to stable is a one-line config change and not a 210 GB reprocess.

In [ ]:
cfg_ds = copy.deepcopy(cfg)
if not (times[0] <= pd.Timestamp(cfg["realtime_start"]) <= times[-1]):
    cfg_ds["realtime_start"] = None          # this subset does not span the boundary

ds = B.build_virtual_dataset(cfg_ds, {STREAM: (arr, times, attrs)}, vds[0])
ds

## 5. Write to a **local** repository

Never point a test at the production store. `local_repo` writes to a throwaway
directory; the references inside it still point at the real netCDFs in GCS.

`local_source_dir` authorizes a second virtual chunk container over the local
block directory. A store written that way is for testing only — its `file://`
references mean nothing to anyone else.

In [ ]:
repo_dir = os.path.join(SCRATCH, "smoke_repo")
shutil.rmtree(repo_dir, ignore_errors=True)
os.makedirs(SCRATCH, exist_ok=True)

repo = B.open_repo(cfg_ds, local_repo=repo_dir, local_source_dir=LOCAL_BLOCK_DIR)
snapshot = B.write_store(repo, ds, "smoke test")

on_disk = sum(os.path.getsize(os.path.join(r, f))
              for r, _, fs in os.walk(repo_dir) for f in fs)
referenced = ds[VAR].size * 4 / 1e9
print(f"committed {snapshot}")
print(f"store on disk: {on_disk / 1e6:.2f} MB, referencing {referenced:.0f} GB of data")

## 6. Read it back the way a consumer will

Not from the writer's in-memory objects — from a fresh read-only session. This is
the only check that proves what was actually persisted.

In [ ]:
store = repo.readonly_session("main").store
out = xr.open_zarr(store, consolidated=False, chunks={})
print(dict(out.sizes))
print(f"time {str(out.time.values[0])[:10]} -> {str(out.time.values[-1])[:10]}")
out

## 7. Compare against the source netCDFs

Structure being right does not mean the bytes are. Check the **first and last time
step of every file**, at one pressure level — and in particular both ends of the
padded tail block, which is the part stock tooling refuses to build and therefore
the part most likely to be wrong.

In [ ]:
import gcsfs
fs = gcsfs.GCSFileSystem(token=B.GCS_TOKEN)
failures = 0

for u in selected:
    if u.startswith("gs://"):
        handle = fs.open(u.replace("gs://", ""), "rb", block_size=8 * 1024 * 1024)
    else:
        handle = u.replace("file://", "")
    src = xr.open_dataset(handle, engine="h5netcdf")
    for t in (src.time.values[0], src.time.values[-1]):
        a = src[VAR].sel(time=t).isel(mean_pressure=0).values
        b = out[VAR].sel(time=t).isel(mean_pressure=0).values
        ok = np.array_equal(a, b, equal_nan=True)
        failures += not ok
        print(f"  {str(t)[:10]}  {'match' if ok else 'MISMATCH'}   {u.split('/')[-1]}")

print("\nSMOKE TEST:", "PASS" if failures == 0 else f"FAIL ({failures} mismatches)")

## 8. How a consumer reads the published store

Both halves need configuring: the **repository** is read from the bucket, and the
**virtual references** are read from wherever the netCDFs live. They are the same
public bucket here, but they are still two separate settings, and a reader that
configures only the first gets metadata and no data.

```python
import icechunk as ic, xarray as xr, zarr

zarr.config.set({"async.concurrency": 128})

storage = ic.gcs_storage(bucket="noaa-oar-rfrom", prefix="icechunk/v2.3", anonymous=True)
prefix  = "gs://noaa-oar-rfrom/netcdf/v2.3/"
repo = ic.Repository.open(
    storage,
    authorize_virtual_chunk_access=ic.containers_credentials(
        {prefix: ic.gcs_credentials(anonymous=True)}),
)
ds = xr.open_zarr(repo.readonly_session("main").store, consolidated=False, chunks={})
```

One caveat to pass on to users: these files are gzip + shuffle, which VirtualiZarr
maps to **numcodecs** codecs. Those are not part of the Zarr v3 core spec, so the
store reads cleanly from zarr-python but may not open in other Zarr
implementations.

## 9. Building the real store

Only after this notebook passes:

```bash
python build_icechunk.py --store rfrom_v23 --list          # what would be referenced
python build_icechunk.py --store rfrom_v23 --local-repo /tmp/rehearsal   # full dry run
python build_icechunk.py --store rfrom_v23                 # write to the bucket
python build_icechunk.py --store rfrom_v23 --validate      # re-check a published store
```